# Carga del Data Warehouse en MySQL

Carga los seis Parquet generados por `load_CSV.ipynb`.

ejemplo `.env` 

```dotenv
MYSQL_HOST=127.0.0.1
MYSQL_PORT=3306
MYSQL_USER=tu_usuario
MYSQL_PASSWORD=tu_contraseña
MYSQL_DATABASE=chobani_dw
```

Si aparece `1045 Access denied`, PyMySQL ya está instalado y sí alcanza el servidor: MySQL rechazó usuario/contraseña o el método de autenticación. Confirma los valores de `.env` (sin compartirlos) y ejecuta de nuevo desde la primera celda. La configuración de este notebook dará prioridad a `.env` frente a variables heredadas del entorno.

Para evitar usar `root`, crea la base y un usuario de carga desde una sesión administrativa de MySQL, sustituyendo la contraseña de ejemplo por una segura. El notebook se conecta a una base ya creada; el usuario de carga no necesita permisos globales para crear bases:

```sql
CREATE DATABASE IF NOT EXISTS chobani_dw CHARACTER SET utf8mb4 COLLATE utf8mb4_unicode_ci;
CREATE USER 'chobani_loader'@'localhost' IDENTIFIED BY 'CAMBIA_ESTA_CONTRASENA';
GRANT SELECT, INSERT, UPDATE, DELETE, CREATE, ALTER, INDEX, REFERENCES
ON chobani_dw.* TO 'chobani_loader'@'localhost';
```

Luego configura `MYSQL_USER=chobani_loader` y la misma contraseña en `.env`. Si conectas por TCP desde otra máquina, reemplaza `localhost` en `CREATE USER`/`GRANT` por el host que MySQL verá para el cliente; limita ese host en vez de abrir el acceso a `%`.

In [1]:
import os
import re
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv
from sqlalchemy import URL, create_engine, text
from sqlalchemy.exc import OperationalError

cwd = Path.cwd().resolve()
PROJECT_ROOT = next(
    (path for path in (cwd, *cwd.parents) if (path / "data" / "processed").is_dir()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "No se encontró data/processed. Ejecuta este notebook desde el proyecto que contiene los Parquet."
    )

ENV_PATH = PROJECT_ROOT / ".env"
load_dotenv(ENV_PATH, override=True)

MYSQL_HOST = os.getenv("MYSQL_HOST", "127.0.0.1")
MYSQL_PORT = int(os.getenv("MYSQL_PORT", "3306"))
MYSQL_USER = os.getenv("MYSQL_USER")
MYSQL_PASSWORD = os.getenv("MYSQL_PASSWORD")
MYSQL_DATABASE = os.getenv("MYSQL_DATABASE", "chobani_dw")

if not MYSQL_USER or not MYSQL_PASSWORD:
    raise ValueError(
        f"Define MYSQL_USER y MYSQL_PASSWORD  o en las variables de entorno."
    )
if not re.fullmatch(r"[A-Za-z0-9_]+", MYSQL_DATABASE):
    raise ValueError("MYSQL_DATABASE solo puede contener letras, números y guion bajo.")

PARQUET_DIR = PROJECT_ROOT / "data" / "processed"
print(f"Proyecto: {PROJECT_ROOT}")

Proyecto: /Users/orozcos/Desktop/Tests/Chobani Test/Take-Home-Test-Nov-26


In [2]:
TABLE_FILES = {
    "dim_clientes": "dim_clientes.parquet",
    "dim_productos": "dim_productos.parquet",
    "fact_pedidos": "fact_pedidos.parquet",
    "fact_lineas_pedido": "fact_lineas_pedido.parquet",
    "quar_pedidos": "quar_pedidos.parquet",
    "quar_lineas_pedido": "quar_lineas_pedido.parquet",
}

EXPECTED_COLUMNS = {
    "dim_clientes": ["customer_id", "customer_name", "region", "channel", "signup_date"],
    "dim_productos": ["product_id", "product_name", "category", "unit_price", "is_active"],
    "fact_pedidos": ["order_id", "customer_id", "order_date", "status", "order_total", "cancel_date", "total_from_lines", "diff_total", "lines_match_total"],
    "fact_lineas_pedido": ["line_id", "order_id", "product_id", "quantity", "unit_price", "discount_pct", "line_amount"],
    "quar_pedidos": ["order_id", "customer_id", "order_date", "status", "order_total", "cancel_date", "order_date_raw", "cancel_date_raw", "motivo", "fecha_deteccion"],
    "quar_lineas_pedido": ["line_id", "order_id", "product_id", "quantity", "unit_price", "discount_pct", "line_amount", "motivo", "fecha_deteccion"],
}

dataframes = {}
for table_name, file_name in TABLE_FILES.items():
    parquet_path = PARQUET_DIR / file_name
    if not parquet_path.is_file():
        raise FileNotFoundError(f"No existe el Parquet requerido: {parquet_path}")
    dataframe = pd.read_parquet(parquet_path, engine="pyarrow")
    missing_columns = set(EXPECTED_COLUMNS[table_name]) - set(dataframe.columns)
    extra_columns = set(dataframe.columns) - set(EXPECTED_COLUMNS[table_name])
    if missing_columns or extra_columns:
        raise ValueError(
            f"Columnas inesperadas en {file_name}. Faltan: {sorted(missing_columns)}; "
            f"sobran: {sorted(extra_columns)}"
        )
    dataframes[table_name] = dataframe[EXPECTED_COLUMNS[table_name]]

print("Parquet listos para cargar:")
for table_name, dataframe in dataframes.items():
    print(f"  {table_name}: {len(dataframe):,} filas")

Parquet listos para cargar:
  dim_clientes: 200 filas
  dim_productos: 40 filas
  fact_pedidos: 4,950 filas
  fact_lineas_pedido: 16,157 filas
  quar_pedidos: 50 filas
  quar_lineas_pedido: 400 filas


In [3]:
TABLE_DDL = [
    """CREATE TABLE IF NOT EXISTS dim_clientes (
        customer_id VARCHAR(16) NOT NULL,
        customer_name VARCHAR(255) NOT NULL,
        region VARCHAR(100) NOT NULL,
        channel VARCHAR(100) NOT NULL,
        signup_date DATE NULL,
        PRIMARY KEY (customer_id)
    ) ENGINE=InnoDB DEFAULT CHARSET=utf8mb4""",
    """CREATE TABLE IF NOT EXISTS dim_productos (
        product_id VARCHAR(16) NOT NULL,
        product_name VARCHAR(255) NOT NULL,
        category VARCHAR(150) NOT NULL,
        unit_price DECIMAL(12,2) NOT NULL,
        is_active BOOLEAN NOT NULL,
        PRIMARY KEY (product_id)
    ) ENGINE=InnoDB DEFAULT CHARSET=utf8mb4""",
    """CREATE TABLE IF NOT EXISTS fact_pedidos (
        order_id VARCHAR(16) NOT NULL,
        customer_id VARCHAR(16) NOT NULL,
        order_date DATE NULL,
        status VARCHAR(32) NOT NULL,
        order_total DECIMAL(14,2) NOT NULL,
        cancel_date DATE NULL,
        total_from_lines DECIMAL(14,2) NOT NULL,
        diff_total DECIMAL(14,2) NOT NULL,
        lines_match_total BOOLEAN NOT NULL,
        PRIMARY KEY (order_id),
        CONSTRAINT fk_fact_pedidos_cliente FOREIGN KEY (customer_id)
            REFERENCES dim_clientes (customer_id)
    ) ENGINE=InnoDB DEFAULT CHARSET=utf8mb4""",
    """CREATE TABLE IF NOT EXISTS fact_lineas_pedido (
        line_id VARCHAR(16) NOT NULL,
        order_id VARCHAR(16) NOT NULL,
        product_id VARCHAR(16) NOT NULL,
        quantity INT NOT NULL,
        unit_price DECIMAL(12,2) NOT NULL,
        discount_pct DECIMAL(6,4) NOT NULL,
        line_amount DECIMAL(14,2) NOT NULL,
        PRIMARY KEY (line_id),
        CONSTRAINT fk_fact_lineas_pedido FOREIGN KEY (order_id)
            REFERENCES fact_pedidos (order_id),
        CONSTRAINT fk_fact_lineas_producto FOREIGN KEY (product_id)
            REFERENCES dim_productos (product_id)
    ) ENGINE=InnoDB DEFAULT CHARSET=utf8mb4""",
    """CREATE TABLE IF NOT EXISTS quar_pedidos (
        quarantine_id BIGINT NOT NULL AUTO_INCREMENT,
        order_id VARCHAR(16) NOT NULL,
        customer_id VARCHAR(16) NULL,
        order_date DATE NULL,
        status VARCHAR(32) NOT NULL,
        order_total DECIMAL(14,2) NOT NULL,
        cancel_date DATE NULL,
        order_date_raw VARCHAR(64) NULL,
        cancel_date_raw VARCHAR(64) NULL,
        motivo VARCHAR(255) NOT NULL,
        fecha_deteccion DATE NULL,
        PRIMARY KEY (quarantine_id),
        INDEX idx_quar_pedidos_order_id (order_id)
    ) ENGINE=InnoDB DEFAULT CHARSET=utf8mb4""",
    """CREATE TABLE IF NOT EXISTS quar_lineas_pedido (
        quarantine_id BIGINT NOT NULL AUTO_INCREMENT,
        line_id VARCHAR(16) NOT NULL,
        order_id VARCHAR(16) NOT NULL,
        product_id VARCHAR(16) NOT NULL,
        quantity INT NOT NULL,
        unit_price DECIMAL(12,2) NOT NULL,
        discount_pct DECIMAL(6,4) NOT NULL,
        line_amount DECIMAL(14,2) NOT NULL,
        motivo VARCHAR(255) NOT NULL,
        fecha_deteccion DATE NULL,
        PRIMARY KEY (quarantine_id),
        INDEX idx_quar_lineas_line_id (line_id),
        INDEX idx_quar_lineas_order_id (order_id)
    ) ENGINE=InnoDB DEFAULT CHARSET=utf8mb4""",
]

In [4]:
database_url = URL.create(
    "mysql+pymysql",
    username=MYSQL_USER,
    password=MYSQL_PASSWORD,
    host=MYSQL_HOST,
    port=MYSQL_PORT,
    database=MYSQL_DATABASE,
    query={"charset": "utf8mb4"},
)
engine = create_engine(database_url, pool_pre_ping=True)

try:
    with engine.begin() as connection:
        for statement in TABLE_DDL:
            connection.execute(text(statement))
except OperationalError as exc:
    error_code = getattr(exc.orig, "args", (None,))[0]
    if error_code == 1045:
        raise RuntimeError(
            f"MySQL rechazó la autenticación para {MYSQL_USER!r} en "
            f"{MYSQL_HOST}:{MYSQL_PORT}. Verifica usuario/contraseña en "
            f".env ejecuta de nuevo desde la primera celda. "
            "No imprimas ni compartas MYSQL_PASSWORD."
        ) from exc
    if error_code == 1049:
        raise RuntimeError(
            f"La base {MYSQL_DATABASE!r} no existe. Créala con una cuenta "
            "administrativa como se indica al inicio del notebook."
        ) from exc
    raise

print(f"Base existente y tablas verificadas en MySQL: {MYSQL_DATABASE}")

Base existente y tablas verificadas en MySQL: chobani_dw


In [5]:
# Recarga de instantánea: DELETE conserva las tablas y sus claves foráneas,
# y permite que toda la sustitución se confirme o revierta en una transacción.
DELETE_ORDER = [
    "quar_lineas_pedido",
    "quar_pedidos",
    "fact_lineas_pedido",
    "fact_pedidos",
    "dim_productos",
    "dim_clientes",
]
LOAD_ORDER = [
    "dim_clientes",
    "dim_productos",
    "fact_pedidos",
    "fact_lineas_pedido",
    "quar_pedidos",
    "quar_lineas_pedido",
]

with engine.begin() as connection:
    for table_name in DELETE_ORDER:
        connection.execute(text(f"DELETE FROM `{table_name}`"))

    for table_name in LOAD_ORDER:
        dataframe = dataframes[table_name]
        dataframe.to_sql(
            name=table_name,
            con=connection,
            if_exists="append",
            index=False,
            chunksize=1000,
            method="multi",
        )
        print(f"Cargada {table_name}: {len(dataframe):,} filas")

print("Carga confirmada.")

Cargada dim_clientes: 200 filas
Cargada dim_productos: 40 filas
Cargada fact_pedidos: 4,950 filas
Cargada fact_lineas_pedido: 16,157 filas
Cargada quar_pedidos: 50 filas
Cargada quar_lineas_pedido: 400 filas
Carga confirmada.


In [6]:
with engine.connect() as connection:
    for table_name in LOAD_ORDER:
        count = connection.execute(text(f"SELECT COUNT(*) FROM `{table_name}`")).scalar_one()
        expected = len(dataframes[table_name])
        if count != expected:
            raise RuntimeError(
                f"Conteo inesperado en {table_name}: MySQL={count}, Parquet={expected}"
            )
        print(f"Verificado {table_name}: {count:,} filas")

engine.dispose()

Verificado dim_clientes: 200 filas
Verificado dim_productos: 40 filas
Verificado fact_pedidos: 4,950 filas
Verificado fact_lineas_pedido: 16,157 filas
Verificado quar_pedidos: 50 filas
Verificado quar_lineas_pedido: 400 filas
